## Initialising ChipWhisperer Lite 

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [ ]:
#Path might be changed if downloaded from GitHub
#I
%run "../../Setup_Scripts/Setup_Generic.ipynb" 

In [ ]:
def reboot_flush():
    reset_target(scope)
    #Flush garbage too
    target.flush()

## Verifying password check output

In [ ]:
#Do glitch loop
pw = bytearray([0x00]*5)
target.simpleserial_write(0x01, pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)
#print(bytearray(val['full_response'].encode('latin-1')))

In [ ]:
#Do glitch loop
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68]) # correct password ASCII representation
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)
print(pw)

## Set up glitch

In [ ]:
scope.cglitch_setup()

In [ ]:
#Check if hs2 changed from clkgen to glitch, if not run the cell below
print(scope.io.hs2)

In [ ]:
scope.io.hs2 = "glitch"
time.sleep(0.2)
reboot_flush()

## Set up glitch statistics and plot

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

## Glitch loop

In [ ]:
def recover_from_hang():
    # make clock stable first (no glitch)
    scope.io.hs2 = "clkgen"
    time.sleep(0.05)

    reboot_flush()   # your improved NRST-based reboot_flush

    # go back to glitch output for the sweep
    scope.io.hs2 = "glitch"
    time.sleep(0.05)

In [ ]:
from tqdm.notebook import tqdm
import re
import struct
import time

sample_size = 1


gc.set_global_step(1)
if PLATFORM == "CWLITEXMEGA":
    #Modify the parameters
        gc.set_range("width", 0, 10) 
        gc.set_range("offset", -10,-10 )
        gc.set_range("ext_offset", 1, 100  )
        gc.set_step("ext_offset", 1)
        gc.set_step("width", 0.5)
        gc.set_step("offset", 0.5)
        #gc.set_global_step(0.2) 
        sample_size = 20

#gc.set_step("ext_offset", 1)
scope.glitch.repeat = 1
reboot_flush()
broken = False
scope.adc.timeout = 1

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    for i in range(sample_size):
        if scope.adc.state:
            # can detect crash here (fast) before timing out (slow)
            print("Trigger still high!")
            gc.add("reset")
            recover_from_hang()
            time.sleep(0.05)
            continue
            #Device is slow to boot?
            reboot_flush()

        scope.arm()
        target.simpleserial_write(0x01, bytes([0]*5))

        ret = scope.capture()

        if ret:
            print('Timeout - no trigger')
            gc.add("reset")

            #Device is slow to boot?
            reboot_flush()

        else:
            val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
            if val['valid'] is False:
                gc.add("reset")
            else:

                if val['payload'] == bytearray([1]): #for loop check
                    #broken = True
                    gc.add("success")
                    print(val['payload'])
                    print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                    print("✅", end="")
                    #break
                else:
                    gc.add("normal")
    if broken:
        break

## Results of most effective parameters

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

In [ ]:
results = gc.calc(ignore_params=["ext_offset"], sort="success_rate")
results

## Check for most reset rates

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="reset_rate")
results

In [ ]:
results = gc.calc(ignore_params=["ext_offset"], sort="reset_rate")
results

In [ ]:
scope.dis()
target.dis()

## Additional codes 

In [ ]:
print(target.baud)

In [ ]:
print(SS_VER)